# Transaction and overdraft evaluation — after your first generation

**If you have run no experiments, run [prepare_account_sample.ipynb](prepare_account_sample.ipynb), then [00_baseline.ipynb](00_baseline.ipynb).** This notebook evaluates its saved baseline and subsequent completed experiments on CPU. It requires no fixed run names and skips evaluation cleanly when there are no results.

One detector learns negative-balance detection from real accounts, then evaluates separate real accounts and generated transactions. Balance creates the label and is excluded from predictors. No loan classification is used. You execute this notebook after creating synthetic data.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image
from IPython.display import display as show


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/transaction-evaluation-matplotlib")
MULTI_TABLE = ROOT / "implementations/tabular_data/multi_table"

from implementations.tabular_data.multi_table.evaluation.transaction_evaluation import evaluate_saved_runs
from implementations.tabular_data.multi_table.experiments.dataset_settings import DATA_DIR as DEFAULT_DATA_DIR
from implementations.tabular_data.multi_table.experiments.tuning import (
    DEFAULT_EXPERIMENT_ROOT,
    DEFAULT_REPORT_ROOT,
    completed_runs,
)


## 1. Find your completed runs

Only the shared `transaction_tuning_from_scratch` workflow directory is searched. No historical guidance runs or old model checkpoints are assumed. Dataset identity and artifact checksums are checked; partial runs are excluded.

In [ ]:
# Edit dataset_settings.py once to switch every notebook, or override here.
DATA_DIR = DEFAULT_DATA_DIR
EXPERIMENT_ROOT = DEFAULT_EXPERIMENT_ROOT
OUTPUT_ROOT = DEFAULT_REPORT_ROOT
BOOTSTRAP_REPETITIONS = 200
EVALUATION_SEED = 42
THRESHOLD = 0.5
RUN_DIRS = completed_runs(DATA_DIR, EXPERIMENT_ROOT) if DATA_DIR.is_dir() else []
comparison = None
REPORT_DIR = None
print(f"Completed runs found: {len(RUN_DIRS)}")
if not RUN_DIRS:
    print("No experiments have completed yet. Prepare the account cohort and run 00_baseline.ipynb; return here after generation.")

print(f"Selected processed dataset: {DATA_DIR}")


## 2. Evaluate saved synthetic data on CPU

The same real account split and detector are used for every selected run. A single baseline is enough to evaluate fidelity; at least one additional variant is needed to compare hyperparameters. Reports include event counts, source/settings fingerprints, fixed predictions, detailed CSVs, and plots.

In [ ]:
if RUN_DIRS:
    comparison, REPORT_DIR = evaluate_saved_runs(
        DATA_DIR, RUN_DIRS, OUTPUT_ROOT, bootstrap_repetitions=BOOTSTRAP_REPETITIONS,
        seed=EVALUATION_SEED, threshold=THRESHOLD,
    )
    print(f"Report saved to: {REPORT_DIR}")
    show(pd.read_json(REPORT_DIR / "real_split_counts.json"))
else:
    print("Evaluation skipped: create a baseline in 00_baseline.ipynb first.")


## 3. Overdraft transfer and uncertainty

AP describes how well overdrafts appear near the top of the scores; report prevalence alongside it. Standardized AP controls only the class-prior difference. ROC-AUC is complementary. Fixed-threshold precision/recall use 0.5 without tuning on these outcomes.

The 95% account-bootstrap intervals condition on the fitted detector and saved generation. Check event counts and valid bootstrap draws; missing/one-class scores are not good results. Repeated generation/training seeds address different uncertainty. This real→synthetic analysis does not establish synthetic-trained generalization to independent real accounts.

In [ ]:
if comparison is not None:
    columns = ["rows", "overdrafts", "overdraft_prevalence", "average_precision",
               "average_precision_low", "average_precision_high", "standardized_ap",
               "roc_auc", "precision", "recall", "bootstrap_valid"]
    show(comparison[columns].round(4))
    show(Image(filename=str(REPORT_DIR / "transfer_intervals.png")))
    show(Image(filename=str(REPORT_DIR / "precision_recall.png")))


## 4. Transaction quality and account relationships

Lower distances indicate closer distributions; valid links and unique primary keys should have rates of 1. Numeric KS covers date, amount, and balance. Categorical TVD covers transaction categories. Identifiers, including external counterparty numbers, are excluded from those measurements.

Overdraft-conditional diagnostics expose rare-event problems hidden by overall marginals. Account checks include counts (including zero-transaction accounts), volumes, mixed parent/child relationships, integrity, and dates aligned to their different epochs. Incomplete histories limit rolling and sequential-balance checks. Full real input data is the fidelity reference.

**Full-table quality is evaluated in the same call:** all declared tables, all direct parent/child edges, PK/FK integrity, cardinality including zero-child parents, distributions, within/cross-table mixed correlations, and client/account combinations through `disp`. Detailed tables are saved as `all_table_scores.csv`, `all_edge_scores.csv`, and `multi_parent_scores.csv`. Undefined scores are reported explicitly.

In [ ]:
if comparison is not None:
    columns = ["transaction_ks", "transaction_tvd", "overdraft_numerical_ks",
               "overdraft_categorical_tvd", "transactions_per_account_ks",
               "account_transaction_correlation_rmse", "accounts_overdrawn_rate_gap",
               "transaction_fk_valid_rate", "transaction_pk_unique_rate",
               "synthetic_transaction_before_account_rate", "max_invalid_rate",
               "max_unknown_category_rate", "undefined_account_pairs"]
    show(comparison.loc[comparison.index != "real_test", columns].round(4))
    show(Image(filename=str(REPORT_DIR / "quality_comparison.png")))

if comparison is not None:
    FULL_TABLE_COLUMNS = [
        "all_table_count", "all_edge_count", "all_table_numerical_ks", "all_table_categorical_tvd",
        "all_cardinality_ks", "all_relationship_correlation_rmse", "all_min_fk_valid_rate",
        "all_min_pk_unique_rate", "multi_parent_correlation_rmse", "multi_parent_joint_categorical_tvd",
        "all_undefined_distribution_columns", "all_undefined_correlation_pairs",
    ]
    print("Full multi-table quality — separate from overdraft transfer:")
    show(comparison.loc[comparison.index != "real_test", FULL_TABLE_COLUMNS].round(4))
    show(Image(filename=str(REPORT_DIR / "multi_table_quality.png")))


## 5. Inspect a completed run

Category event rates include support and event counts; small groups are uncertain. Inspect overdraft-only distributions, missing/unknown values, and chronology before selecting a parameter. With no completed runs this cell does nothing.

In [ ]:
if REPORT_DIR is not None:
    INSPECT_RUN = RUN_DIRS[0].name  # Change to another completed run name if desired.
    DETAIL_DIR = REPORT_DIR / INSPECT_RUN
    show(pd.read_csv(DETAIL_DIR / "distributions.csv").round(4))
    show(pd.read_csv(DETAIL_DIR / "conditional_distributions.csv").query("overdraft == 1").round(4))
    groups = pd.read_csv(DETAIL_DIR / "overdraft_by_category.csv")
    show(groups.sort_values("rate_error", ascending=False).head(12).round(4))

if REPORT_DIR is not None:
    show(pd.read_csv(DETAIL_DIR / "all_table_scores.csv").round(4))
    show(pd.read_csv(DETAIL_DIR / "all_edge_scores.csv").round(4))
    show(pd.read_csv(DETAIL_DIR / "multi_parent_scores.csv").round(4))


## Conclusion — fill with your results

1. **Baseline behavior:** … Include event counts, prevalence, AP/uncertainty, and the real holdout comparison.
2. **Problems and tradeoffs:** … Include overdraft-specific distributions, account relationships, and chronology.
3. **Next experiment:** … Choose one parameter and keep a matched baseline.

No previous experimental results are assumed. A single generated baseline establishes a reference, not a hyperparameter winner. Use [all_experiments.ipynb](all_experiments.ipynb) for configurations alongside the final cross-experiment comparison.

Include both evaluation views in the final decision. These scores compare to the selected account cohort, not to the entire unsampled population. Use the same DATA_DIR and baseline when attributing a difference to a hyperparameter.